# IVPFGraph: Xining baseline (13 objects)

Source: Amanathulla and Khatun (2025), Journal of Fuzzy Extension and Applications 6(2), 284–299.

Adaptation: Constructs interval-valued picture fuzzy inputs from the GNN values and expert spread; uses the graph aggregation and score construction with six attributes.

Settings: range-based intervals; neutral scale=1; radius scale=0.5 (cap=0.25); edge-uncertainty scale=0.5.

The small input arrays are embedded below. See README.md for shared settings, execution, and interpretation. Parameters were selected on these same objects using CAS.


In [ ]:
import hashlib
import json
from itertools import combinations

import numpy as np

# Fixed 13-object input snapshot; no external notebook is required.
values = {'X1': [[232844400, 363040200, 951871500, 16028100, 12839400, 63332100],
        [498796200, 836140500, 1523463300, 7876800, 77425200, 145968300],
        [590792400, 408241800, 1264804200, 8746200, 119119500, 27847800],
        [453535200, 281313000, 2052270900, 45549900, 29835000, 26512200],
        [109103400, 84975300, 478243800, 2700900, 22102200, 0],
        [43988400, 3285900, 40133700, 1599300, 48115800, 0],
        [219577500, 331435800, 791136000, 0, 28975500, 45741600],
        [1898100, 186300, 13910400, 53100, 9067500, 0], [6731100, 1264500, 37102500, 827100, 23383800, 0],
        [10149300, 624600, 69009300, 2301300, 32026500, 0],
        [783862200, 172843200, 847346400, 14004000, 24583500, 8267400],
        [480004200, 385567200, 1648658700, 9315900, 40091400, 17743500],
        [638441100, 723882600, 1665636300, 16281900, 81209700, 86106600]],
 'X2': [[234132300, 364105800, 949576500, 15961500, 12839400, 63340200],
        [499432500, 838892700, 1520082900, 7876800, 77425200, 145960200],
        [590792400, 408241800, 1264804200, 8746200, 119119500, 27847800],
        [456410700, 281717100, 2048993100, 45548100, 29835000, 26512200],
        [109103400, 84975300, 478243800, 2700900, 22102200, 0],
        [43988400, 3285900, 40133700, 1599300, 48115800, 0],
        [219577500, 331435800, 791136000, 0, 28975500, 45741600],
        [1898100, 186300, 13910400, 53100, 9067500, 0], [6731100, 1264500, 37102500, 827100, 23383800, 0],
        [10149300, 624600, 69009300, 2301300, 32026500, 0],
        [783862200, 172845900, 847343700, 14004000, 24583500, 8267400],
        [480004200, 385567200, 1648658700, 9315900, 40091400, 17743500],
        [638441100, 723885300, 1665633600, 16281900, 81209700, 86106600]],
 'X3': [[234090900, 363079800, 950717700, 15875100, 12839400, 63352800],
        [499121100, 836076600, 1523223000, 7876800, 77425200, 145947600],
        [590792400, 408241800, 1264804200, 8746200, 119119500, 27847800],
        [456381000, 281336400, 2049408900, 45542700, 29835000, 26512200],
        [109103400, 84975300, 478243800, 2700900, 22102200, 0],
        [43988400, 3285900, 40133700, 1599300, 48115800, 0],
        [219577500, 331435800, 791136000, 0, 28975500, 45741600],
        [1898100, 186300, 13910400, 53100, 9067500, 0], [6731100, 1264500, 37102500, 827100, 23383800, 0],
        [10149300, 624600, 69009300, 2301300, 32026500, 0],
        [783862200, 172844100, 847345500, 14004000, 24583500, 8267400],
        [480004200, 385567200, 1648658700, 9315900, 40091400, 17743500],
        [638441100, 723882600, 1665636300, 16281900, 81209700, 86106600]],
 'X4': [[233073000, 364482000, 950026500, 16191000, 12839400, 63343800],
        [497124900, 842259600, 1519027200, 7876800, 77425200, 145956600],
        [590792400, 408241800, 1264804200, 8746200, 119119500, 27847800],
        [453763800, 281916900, 2051433900, 45554400, 29835000, 26512200],
        [109103400, 84975300, 478243800, 2700900, 22102200, 0],
        [43988400, 3285900, 40133700, 1599300, 48115800, 0],
        [219577500, 331435800, 791136000, 0, 28975500, 45741600],
        [1898100, 186300, 13910400, 53100, 9067500, 0], [6731100, 1264500, 37102500, 827100, 23383800, 0],
        [10149300, 624600, 69009300, 2301300, 32026500, 0],
        [783862200, 172849500, 847340100, 14004000, 24583500, 8267400],
        [480004200, 385567200, 1648658700, 9315900, 40091400, 17743500],
        [638441100, 723893400, 1665625500, 16281900, 81209700, 86106600]],
 'X_GNN': [[1.2847, 0.0328, 0.9199, 0.547, 0.9623, 1.0416], [1.4388, 0.2173, 0.9251, 0.3661, 0.7657, 1.0301],
           [0.5799, 0.4036, 0.5127, 0.3826, 0.3668, 0.6181], [0.6514, 0.0, 0.6561, 0.5955, 0.7861, 0.7097],
           [1.0569, 0.2446, 1.1748, 0.8674, 0.9807, 1.2204], [0.3136, 0.3107, 0.2432, 0.2576, 0.2307, 0.356],
           [1.4631, 0.2302, 1.0372, 0.4263, 0.813, 1.0689], [0.1929, 0.1989, 0.1862, 0.1943, 0.1937, 0.1882],
           [0.3136, 0.3107, 0.2432, 0.2576, 0.2307, 0.356], [0.3136, 0.3107, 0.2432, 0.2576, 0.2307, 0.356],
           [0.39, 0.2622, 0.5323, 0.5137, 0.4475, 0.623], [1.2794, 0.2737, 1.2803, 0.8464, 1.0261, 1.3336],
           [1.5277, 0.3486, 1.2552, 0.6476, 0.9244, 1.339]],
 'w_e': [0.2506, 0.2519, 0.2607, 0.2368],
 'C_chengben': [5]}
DATA_SHA256 = hashlib.sha256(
    json.dumps(values, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()

raw = np.stack([np.asarray(values["X" + str(i)], dtype=float) for i in range(1, 5)])
w_E = np.asarray(values["w_e"], dtype=float)
w_E = w_E / w_E.sum()
cost_idx = tuple(int(j) for j in values["C_chengben"])

# Identical external attribute weights for all six updated baselines.
# The proposed model retains its own attribute-weighting calculation.
w_A = np.asarray([0.247, 0.088, 0.259, 0.166, 0.192, 0.048], dtype=float)
q = 5  # Shared q-rung setting.

assert raw.shape == (4, 13, 6), raw.shape
assert w_E.shape == (4,) and np.isclose(w_E.sum(), 1)
assert w_A.shape == (6,) and abs(w_A.sum() - 1.001) < 1e-12
assert cost_idx == (5,)

# Shared GNN output is the primary 13 x 6 input to all six baselines.
X_GNN = np.asarray(values["X_GNN"], dtype=float)
assert X_GNN.shape == (13, 6)

def FuzzyNorm(X, q, C_chengben, eps=0.05):
    """Convert rows to q-ROF pairs with deterministic boundary treatment."""
    if q < 1 or not 0 < eps < 1:
        raise ValueError("Require q >= 1 and 0 < eps < 1")

    Min = []
    Max = []
    for i in range(len(X)):
        Min.append(min(X[i]))
        Max.append(max(X[i]))

    U = [[] for _ in range(len(X))]
    V = [[] for _ in range(len(X))]
    for i in range(len(X)):
        if Max[i] == Min[i]:
            raise ValueError(f"Object {i+1} has constant attributes; row normalization is undefined")
        for j in range(len(X[0])):
            if j not in C_chengben:
                u = round((X[i][j] - Min[i]) / (Max[i] - Min[i]), 3)
                v = round((1 - u**q)**(1/q), 3)
            else:
                v = round((Max[i] - X[i][j]) / (Max[i] - Min[i]), 3)
                u = round((1 - v**q)**(1/q), 3)
            # Independent three-decimal rounding can cross the q-ROF boundary.
            # Keep the normalized component and lower only the derived one.
            if u**q + v**q > 1 + 1e-12:
                if j not in C_chengben:
                    v = int((max(0.0, 1-u**q)**(1/q))*1000 + 1e-12) / 1000
                else:
                    u = int((max(0.0, 1-v**q)**(1/q))*1000 + 1e-12) / 1000
            U[i].append(u)
            V[i].append(v)

    high = (1 - eps**q)**(1/q)
    if high >= 1.0:
        raise ValueError("The chosen q and eps round high to 1; increase eps")

    for i in range(len(U)):
        for j in range(len(U[0])):
            if U[i][j] == 0.0 and V[i][j] == 1.0:
                U[i][j] = eps
                V[i][j] = high
            elif U[i][j] == 1.0 and V[i][j] == 0.0:
                U[i][j] = high
                V[i][j] = eps

    return U, V

# Retain the four original expert matrices for methods that need expert
# disagreement or expert-level graph/rule structure. Their weights are fixed.
U_GNN, V_GNN = FuzzyNorm(X_GNN, q, cost_idx, eps=0.05)
X = np.asarray(U_GNN, dtype=float)
X_nonmembership = np.asarray(V_GNN, dtype=float)
expert_pairs = [FuzzyNorm(raw[e], q, cost_idx, eps=0.05) for e in range(raw.shape[0])]
utility = np.asarray([pair[0] for pair in expert_pairs], dtype=float)
expert_nonmembership = np.asarray([pair[1] for pair in expert_pairs], dtype=float)
assert utility.shape == (4, 13, 6) and X.shape == (13, 6)
assert np.isfinite(utility).all() and np.isfinite(X).all()
assert np.all((utility >= 0) & (utility <= 1))
assert np.all((X >= 0) & (X <= 1))
assert np.array_equal(X[5], X[8]) and np.array_equal(X[5], X[9])
assert np.array_equal(X_nonmembership[5], X_nonmembership[8]) and np.array_equal(X_nonmembership[5], X_nonmembership[9])

def report_ranking(scores, decimals=12):
    scores = np.asarray(scores, dtype=float)
    if scores.shape != (13,) or not np.isfinite(scores).all():
        raise ValueError("Expected 13 finite scores, got " + repr(scores))
    order = np.argsort(-scores, kind="stable")
    groups = []
    for idx in order:
        if groups and np.isclose(scores[idx], scores[groups[-1][0]], rtol=1e-12, atol=1e-15):
            groups[-1].append(int(idx))
        else:
            groups.append([int(idx)])
    expression = " > ".join(" = ".join("O" + str(i + 1) for i in group) for group in groups)
    print("Final ranking (= denotes a tie):")
    print(expression)
    print("\nScores by object:")
    for i, s in enumerate(scores):
        print("O{:02d}: {:.{}g}".format(i + 1, s, decimals))
    print("\nEmbedded data SHA256:", DATA_SHA256)
    return groups

print("Input: 4 experts, 13 objects, 6 attributes; attribute 6 is cost-type")
print("Expert weights:", w_E.tolist())
print("Shared attribute weights:", w_A.tolist())
print("q =", q)
print("Shared input: X_GNN -> FuzzyNorm(q=5, eps=0.05)")


Input: 4 experts, 13 objects, 6 attributes; attribute 6 is cost-type
Expert weights: [0.25060000000000004, 0.25190000000000007, 0.26070000000000004, 0.23680000000000004]
Shared attribute weights: [0.285, 0.066, 0.301, 0.137, 0.152, 0.06]
q = 5
Shared input: X_GNN -> FuzzyNorm(q=5, eps=0.05)


## Method computation


In [2]:
# Data-based interval-valued picture fuzzy numbers for each alternative/attribute.
spread = utility.max(axis=0) - utility.min(axis=0)
neutral = np.minimum(spread, 0.5)
base = np.stack([(1-neutral)*X, neutral, (1-neutral)*(1-X)], axis=2)
radius = np.minimum(0.5*spread, 0.25)
zeta_lower = base * (1-radius[:,:,None])
zeta_upper = base.copy()
assert np.all(zeta_lower <= zeta_upper + 1e-15)
assert np.all(zeta_upper.sum(axis=2) <= 1 + 1e-12)

# IVPF attribute-edge relation from correlations of expert-level attribute profiles.
delta = np.eye(6)
edge_lower = np.zeros((6,6,3)); edge_upper = np.zeros_like(edge_lower)
for j,k in combinations(range(6),2):
    correlations = np.array([np.corrcoef(utility[e,:,j], utility[e,:,k])[0,1]
                             for e in range(4)])
    correlations = np.nan_to_num(correlations)
    similarity = np.clip((1 + np.dot(w_E, correlations))/2, 0, 1)
    uncertainty = np.clip(0.5*np.sqrt(np.dot(w_E,
                            (correlations - np.dot(w_E,correlations))**2)), 0, 0.5)
    edge = np.array([(1-uncertainty)*similarity, uncertainty,
                     (1-uncertainty)*(1-similarity)])
    width = min(0.25, uncertainty/2)
    lower_edge = edge*(1-width)
    edge_lower[j,k] = edge_lower[k,j] = lower_edge
    edge_upper[j,k] = edge_upper[k,j] = edge
    p_l,n_l,f_l = lower_edge
    p_u,n_u,f_u = edge
    delta[j,k] = delta[k,j] = ((p_l+p_u) + (2-n_l-n_u)*(2-f_l-f_u))/6
assert np.all((delta >= 0) & (delta <= 1))
assert np.all(edge_upper.sum(axis=2) <= 1 + 1e-12)

# Paper Algorithm 2 Step 2, with 1/n instead of 1/3 for six attributes.
coeff = delta @ w_A / X.shape[1]
agg_lower = np.einsum("ijc,j->ic", zeta_lower, coeff)
agg_upper = np.einsum("ijc,j->ic", zeta_upper, coeff)
assert np.all(agg_upper.sum(axis=1) <= 1 + 1e-12)
p_l,n_l,f_l = agg_lower.T
p_u,n_u,f_u = agg_upper.T
scores = (2 + p_l+p_u - 2*n_l-2*n_u - f_l-f_u)/4
print("Off-diagonal influence-coefficient range:",
      float(delta[np.triu_indices(6,1)].min()), float(delta[np.triu_indices(6,1)].max()))
print("Algorithm 2 score; aggregation coefficient extended to 1/n for six attributes")


Off-diagonal influence-coefficient range: 0.15476773539847025 0.9642784974624705
Algorithm 2 score; aggregation coefficient extended to 1/n for six attributes


## Ranking and scores


In [3]:
rank_groups = report_ranking(scores)


Final ranking (= denotes a tie):
O4 > O5 > O12 > O1 > O8 > O13 > O11 > O7 > O2 > O6 = O9 = O10 > O3

Scores by object:
O01: 0.578179799114
O02: 0.516446486181
O03: 0.476455982674
O04: 0.626899104337
O05: 0.607446095607
O06: 0.480506337226
O07: 0.528901130888
O08: 0.571218157415
O09: 0.480506337226
O10: 0.480506337226
O11: 0.532692395813
O12: 0.605186905863
O13: 0.549281975944

Embedded data SHA256: 915a03f9bef24a3f651d3398d3a590e28027949269ab027c0c1096aef4db92a5
